# mPES — Optimización Bayesiana en Colab Pro+

Ejecuta optimización Bayesiana de **pes_dqn** o **pes_ac** en Colab Pro+.
La DB de Optuna se respalda en Drive y se restaura automáticamente al reconectar.

> Los modelos son pequeños (32–64 neuronas). La GPU no acelera mucho,
> pero la instancia GPU trae más RAM y mejor CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title **Configuración** { display-mode: "form" }

PACKAGE = "pes_dqn" #@param ["pes_dqn", "pes_ac"]
N_TRIALS = 100 #@param {type:"integer"}
RESUME_DATE = "2026-03-07" #@param {type:"string"}
BRANCH = "dqn_and_ac" #@param {type:"string"}
DRIVE_SYNC_MINUTES = 15 #@param {type:"slider", min:1, max:30, step:1}
GITHUB_TOKEN = "Token_2026" #@param {type:"string"}
NTFY_TOPIC = "mpes_opt" #@param {type:"string"}
NTFY_EVERY_N = 5 #@param {type:"integer"}

# --- Derivados ---
import os

GITHUB_REPO = 'https://github.com/Maximiliano0/mPES_2026.git'
_MODULE_MAP = {'pes_dqn': 'optimize_dqn', 'pes_ac': 'optimize_ac'}
OPT_MODULE = _MODULE_MAP[PACKAGE]
REPO_DIR = '/content/mPES'
DRIVE_BACKUP = f'/content/drive/MyDrive/mPES_backups/{PACKAGE}'
DB_SUBDIR = f'{PACKAGE}/inputs/{RESUME_DATE}_BAYESIAN_OPT'
DB_NAME = f'optuna_study_{RESUME_DATE}.db'

os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f'Paquete: {PACKAGE}  |  Módulo: {PACKAGE}.ext.{OPT_MODULE}')
print(f'Trials: {N_TRIALS}  |  Resume: {RESUME_DATE}  |  Rama: {BRANCH}')
print(f'Backup: {DRIVE_BACKUP}')

## 1. Setup del entorno

In [ ]:
import os, shutil, subprocess

# --- Clonar o actualizar repo ---
if not os.path.exists(REPO_DIR):
    repo_url = GITHUB_REPO
    if GITHUB_TOKEN:
        repo_url = GITHUB_REPO.replace('https://', f'https://{GITHUB_TOKEN}@')
    subprocess.run(['git', 'clone', '--branch', BRANCH, repo_url, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

# --- Restaurar DB: Drive > local > nueva ---
drive_db = os.path.join(DRIVE_BACKUP, DB_NAME)
repo_db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
os.makedirs(os.path.dirname(repo_db), exist_ok=True)

if os.path.exists(drive_db):
    shutil.copy2(drive_db, repo_db)
    print(f'DB restaurada desde Drive')
elif os.path.exists(repo_db):
    shutil.copy2(repo_db, drive_db)
    print('DB local copiada a Drive como backup')
else:
    print('No se encontr\u00f3 DB \u2014 se crear\u00e1 una nueva')

# --- Dependencias ---
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.0',
    'optuna==4.7.0', 'gym==0.26.1', 'pygame==2.5.2', 'rich==14.3.2',
    'mne>=1.6.1', 'scipy>=1.14.0', 'colorlog', 'colorama'
], check=True)

# --- Verificar numpy 2.x (requerido por TF de Colab) ---
import numpy
_nv = numpy.__version__
if int(_nv.split('.')[0]) < 2:
    raise RuntimeError(
        f'numpy {_nv} es 1.x \u2014 reiniciu00e1 el runtime (Runtime \u2192 Restart session) '
        f'y volvu00e9 a ejecutar todas las celdas.'
    )

import tensorflow as tf
import optuna
gpus = tf.config.list_physical_devices('GPU')
print(f'numpy {_nv}, TF {tf.__version__}, Optuna {optuna.__version__}')
print(f'GPU: {gpus[0].name if gpus else "ninguna (CPU)"}')

In [ ]:
import subprocess, os, sqlite3

subprocess.run(['nvidia-smi'], check=False)

with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line:
            print(f'\nRAM total: {int(line.split()[1]) / 1024**2:.1f} GB')
            break

# --- Verificar archivos ---
repo_db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
csv1 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'sequence_lengths.csv')
csv2 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'initial_severity.csv')

print('\nArchivos requeridos:')
for fpath in [repo_db, csv1, csv2]:
    ok = os.path.exists(fpath)
    print(f'  {"OK" if ok else "FALTA"}  {os.path.relpath(fpath, REPO_DIR)}')

# --- Estado de la DB ---
if os.path.exists(repo_db):
    conn = sqlite3.connect(repo_db)
    print('\nEstado de trials:')
    for row in conn.execute('SELECT state, COUNT(*) FROM trials GROUP BY state'):
        print(f'  {row[0]}: {row[1]}')
    best = conn.execute(
        'SELECT MAX(tv.value) FROM trial_values tv '
        'JOIN trials t ON tv.trial_id=t.trial_id WHERE t.state="COMPLETE"'
    ).fetchone()
    if best and best[0]:
        print(f'  Mejor valor: {best[0]:.6f}')
    conn.close()

## 2. Ejecutar optimización

Lanza la optimización como subproceso. Un hilo respalda la DB en Drive cada N minutos.
Si Colab se desconecta, reejecutar todas las celdas restaura la DB y retoma.

In [ ]:
import subprocess, os, threading, time, shutil, re, sqlite3
import urllib.request

db_src = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
db_dst = os.path.join(DRIVE_BACKUP, DB_NAME)


def ntfy(msg, title=None, priority='default', tags=''):
    """Env\u00eda notificaci\u00f3n push v\u00eda ntfy.sh."""
    if not NTFY_TOPIC:
        return
    try:
        headers = {}
        if title:
            safe = title.encode('latin-1', errors='ignore').decode('latin-1').strip()
            if safe:
                headers['Title'] = safe
        if priority and priority != 'default':
            headers['Priority'] = priority
        if tags:
            headers['Tags'] = tags
        req = urllib.request.Request(
            f'https://ntfy.sh/{NTFY_TOPIC}',
            data=msg.encode('utf-8'), headers=headers, method='POST'
        )
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:
        print(f'  [ntfy] Error: {e}', flush=True)


def query_db(db_path):
    """Retorna (n_complete, best_value) desde la DB de Optuna."""
    try:
        conn = sqlite3.connect(db_path)
        n = conn.execute("SELECT COUNT(*) FROM trials WHERE state='COMPLETE'").fetchone()[0]
        best = conn.execute(
            "SELECT MAX(tv.value) FROM trial_values tv "
            "JOIN trials t ON tv.trial_id=t.trial_id WHERE t.state='COMPLETE'"
        ).fetchone()[0]
        conn.close()
        return n, best
    except Exception:
        return 0, None


def fmt_dur(s):
    """Segundos a formato legible."""
    if s < 60: return f'{s:.0f}s'
    if s < 3600: return f'{int(s//60)}m {int(s%60)}s'
    return f'{int(s//3600)}h {int((s%3600)//60)}m'


# --- Hilo de sync a Drive ---
done_event = threading.Event()

def sync_to_drive():
    while not done_event.is_set():
        done_event.wait(DRIVE_SYNC_MINUTES * 60)
        if done_event.is_set():
            break
        try:
            shutil.copy2(db_src, db_dst)
            print(f'  [Drive Sync {time.strftime("%H:%M:%S")}] DB respaldada', flush=True)
        except Exception as e:
            print(f'  [Drive Sync] Error: {e}', flush=True)

threading.Thread(target=sync_to_drive, daemon=True).start()

# --- Entorno y comando ---
env = os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES': '0',
    'VIRTUAL_ENV': '/content',
    'PYTHONUNBUFFERED': '1',
    'TF_ENABLE_ONEDNN_OPTS': '0',
    'TF_CPP_MIN_LOG_LEVEL': '2',
})

cmd = ['python3', '-m', f'{PACKAGE}.ext.{OPT_MODULE}', str(N_TRIALS), '--resume', RESUME_DATE]

# --- Estado inicial y notificaci\u00f3n ---
initial_n, initial_best = query_db(db_src)
ntfy(
    f'Optimizaci\u00f3n iniciada\nPaquete: {PACKAGE}\n'
    f'Trials: {initial_n}/{N_TRIALS}\nMejor: {initial_best or "---"}',
    title=f'{PACKAGE} \u2014 Iniciado', tags='rocket'
)

print(f'\u25b6 {" ".join(cmd)}')
print(f'  cwd: {REPO_DIR}  |  Reanudando desde trial {initial_n}/{N_TRIALS}')
if initial_best:
    print(f'  Mejor valor actual: {initial_best:.6f}')
print()

# --- Ejecutar subproceso ---
completed = [initial_n]
last_notified = [initial_n]
best_value = initial_best
t_start = time.time()
_re_best = re.compile(r'[Bb]est\s+value[:\s]+([0-9]+\.[0-9]+)')

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

for line in proc.stdout:
    m = _re_best.search(line)
    if m:
        best_value = float(m.group(1))

    if re.search(r'[Ff]inished|COMPLETE|complete', line):
        n, val = query_db(db_src)
        if n > 0:
            completed[0] = n
        if val is not None:
            best_value = val

        # Notificaci\u00f3n peri\u00f3dica
        if (completed[0] - last_notified[0]) >= NTFY_EVERY_N:
            last_notified[0] = completed[0]
            elapsed = time.time() - t_start
            eta = (elapsed / max(completed[0], 1)) * (N_TRIALS - completed[0])
            ntfy(
                f'Progreso: {completed[0]}/{N_TRIALS}\n'
                f'Mejor: {best_value:.4f if best_value else "---"}\n'
                f'Tiempo: {fmt_dur(elapsed)}  |  ETA: {fmt_dur(eta)}',
                title=f'{PACKAGE} \u2014 {completed[0]}/{N_TRIALS}',
                tags='chart_with_upwards_trend'
            )

    print(line, end='', flush=True)

# --- Finalizaci\u00f3n ---
exit_code = proc.wait()
done_event.set()
elapsed_total = time.time() - t_start

final_n, final_best = query_db(db_src)
if final_n > 0: completed[0] = final_n
if final_best is not None: best_value = final_best

# Backup final
shutil.copy2(db_src, db_dst)

print(f'\n{"="*50}')
print(f'  FINALIZADO  |  Trials: {completed[0]}/{N_TRIALS}  |  '
      f'Mejor: {best_value:.6f if best_value else "---"}  |  '
      f'Tiempo: {fmt_dur(elapsed_total)}  |  Exit: {exit_code}')
print(f'{"="*50}')

ntfy(
    f'Finalizado\nTrials: {completed[0]}/{N_TRIALS}\n'
    f'Mejor: {best_value:.6f if best_value else "---"}\n'
    f'Tiempo: {fmt_dur(elapsed_total)}\n'
    f'Exit: {exit_code}',
    title=f'{PACKAGE} \u2014 Finalizado',
    priority='high',
    tags='white_check_mark' if exit_code == 0 else 'warning'
)

## 3. Resultados

In [ ]:
import sqlite3, os, shutil

db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
conn = sqlite3.connect(db)

print('Trials por estado:')
for row in conn.execute('SELECT state, COUNT(*) FROM trials GROUP BY state'):
    print(f'  {row[0]}: {row[1]}')

print('\nTop 10 trials:')
rows = conn.execute(
    'SELECT t.trial_id, tv.value '
    'FROM trial_values tv JOIN trials t ON tv.trial_id=t.trial_id '
    'WHERE t.state="COMPLETE" ORDER BY tv.value DESC LIMIT 10'
).fetchall()
for i, row in enumerate(rows, 1):
    marker = ' (BEST)' if i == 1 else ''
    print(f'  #{row[0]:3d}  {row[1]:.6f}{marker}')
conn.close()

# --- Copiar outputs a Drive ---
opt_dir_src = os.path.join(REPO_DIR, DB_SUBDIR)
opt_dir_dst = os.path.join(DRIVE_BACKUP, f'{RESUME_DATE}_BAYESIAN_OPT')
if os.path.exists(opt_dir_src):
    shutil.copytree(opt_dir_src, opt_dir_dst, dirs_exist_ok=True)
    print(f'\nOutputs copiados a: {opt_dir_dst}')
    for fname in sorted(os.listdir(opt_dir_dst)):
        size = os.path.getsize(os.path.join(opt_dir_dst, fname))
        print(f'  {fname} ({size/1024:.0f} KB)')